In [2]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
import pyvinecopulib as pv
from scipy.stats import genpareto, kendalltau
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple, Literal
import numpy as np
import cvxpy as cp
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from sklearn.covariance import LedoitWolf

In [5]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    #super().__init__(debug=debug, **kwargs)
    self.debug = debug

  def _get_data(
      self,
      universe:dict,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    tickers_raw = list(asdict(universe).values())
    tmp = []
    for t in tickers_raw:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)

      else:
        print(f"Warning: Skipping {t} | type: {type(t)} ")

    tickers_clean = list(set(tmp))

    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      if benchmark not in tickers_clean:
        tickers_clean.append(benchmark)

      df = yf.download(tickers_clean, start, end, interval)["Close"]

      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    bench_data = df["^GSPC"]
    data_raw = df.drop(columns=["^GSPC"])

    benchmark = bench_data.pct_change().dropna()
    returns = data_raw.pct_change().dropna()
    self.universe = data_raw.columns

    return returns, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [6]:
returns, _ = DataStore()._get_data(
    universe=test_universe,
    start="2019-05-09",
    end="2026-08-25",
    interval="1d",
    benchmark="^GSPC"
)

/tmp/ipykernel_1611/2783720663.py:34: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers_clean, start, end, interval)["Close"]
[*********************100%***********************]  10 of 10 completed


# Cluster Engine

In [7]:
@dataclass
class ModelParams:
  distance_method:str="kendalltau"
  optimize:bool=True
  cvar_q:float|None=0.95
  use_lw_shrinkage:bool=False
  k_max:int=8
  n_sims:int=100

  def __post_init__(self):
    allowed_methods = {"kendalltau", "pearsoncorr"}
    if self.distance_method not in allowed_methods:
      raise ValueError(f"Invalid status. Choose from {allowed_methods}")

    if not (0 <= self.cvar_q <= 1):
      raise ValueError("Quantile must be between 0 and 1")

In [8]:
@dataclass
class ClusterParams:
  quantiles: dict[str, float] = field(default_factory=dict)
  paths: np.ndarray = None

In [9]:
class ClusterEngine:
  def __init__(self, params: ModelParams, debug=False, **kwargs):
    #super().__init__(params=params, debug=debug, **kwargs)
    self.cp = ClusterParams()
    self.p = params


  def _get_tail_mask(self, sim_returns, assets, w=None, sample=False):
    if w is None:
      w = np.ones(len(assets)) / len(assets)

    asset_R = np.prod(1 + sim_returns, axis=2) - 1
    asset_L = - asset_R
    alpha = self.p.cvar_q * 100

    p_R = asset_L @ w

    var_p = np.percentile(p_R, alpha)
    tail_mask = p_R >= var_p

    return tail_mask

  def _get_tau_dist(self, sim_returns, assets, w=None):
    tail_mask = self._get_tail_mask(sim_returns, assets, w)

    tau_mtx = np.zeros((len(assets), len(assets)))
    tail_returns = sim_returns[tail_mask, :]
    asset_losses = -tail_returns

    for i in range(len(assets)):
      for j in range(len(assets)):
        tau_ij, _ = kendalltau(
          asset_losses[:, i],
          asset_losses[:, j]
        )

        tau_mtx[i, j] = tau_ij

    dist_mtx = np.sqrt(0.5 * (1 - tau_mtx))

    dist_df = pd.DataFrame(columns=assets, index=assets, data=dist_mtx)

    return dist_df

  def _get_dist_mtx(self, returns:np.ndarray|pd.DataFrame, assets=None):
    if self.p.distance_method == "kendalltau":
      assert assets is not None, "Assets must be provided for Kendall Tau"
      dist_mtx = self._get_tau_dist(returns, assets)

    elif self.p.distance_method == "pearsoncorr":
      corr_mtx = np.clip(np.corrcoef(returns, rowvar=False), -1.0, 1.0)
      dist_mtx = np.sqrt(0.5 * (1 - corr_mtx))

    return dist_mtx

  def _generate_null_dists(self, returns_df):
    N_samples, N_assets = returns_df.shape
    returns_arr = returns_df.values
    min_bounds = np.min(returns_arr, axis=0)
    max_bounds = np.max(returns_arr, axis=0)

    null_dists, null_links = [], []

    for i in range(self.p.n_sims):
      null_returns = np.random.uniform(
        low=min_bounds,
        high=max_bounds,
        size=(self.p.n_sims, N_samples, N_assets)
      )

      null_dist = self._get_dist_mtx(
          null_returns.reshape(self.p.n_sims, N_assets, N_samples),
          returns_df.columns
      )

      condensed_dist = squareform(null_dist, checks=False)
      Z_null = linkage(condensed_dist, method="single")

      null_dists.append(null_dist)
      null_links.append(Z_null)

    return np.array(null_dists), np.array(null_links)

  def _compute_cl_dispersion(self, dist_mtx, cluster_labels):
    if isinstance(dist_mtx, pd.DataFrame):
      dist_mtx = dist_mtx.to_numpy()

    unique_clusters = np.unique(cluster_labels)
    W_k = 0.0

    for c_id in unique_clusters:
      cluster_indices = np.where(cluster_labels == c_id)[0]
      cluster_dist = dist_mtx[np.ix_(cluster_indices, cluster_indices)]

      norm = 2*len(cluster_indices)

      D_r = np.sum(cluster_dist**2)

      W_k += D_r / norm

    return W_k

  def _compute_ref_log_disp(self, null_dists, null_links, k):
    B = self.p.n_sims
    W_k_log = []

    for b in range(B):
      clusters = fcluster(null_links[b], t=k, criterion="maxclust")

      W_k = self._compute_cl_dispersion(null_dists[b], clusters)
      W_k_log.append(np.log(max(W_k, 1e-300)))

    W_k_log = np.array(W_k_log)
    E_W_k = np.mean(W_k_log)

    sdk = np.std(W_k_log, ddof=1) if B > 1 else 0.0
    s_k = sdk * np.sqrt(1+1/B)

    return E_W_k, s_k

  def _get_k_clusters(self, dist_mtx, Z, returns):
    k_max = min(self.p.k_max, dist_mtx.shape[0]-1)
    Gap_k, s_k_list = [], []
    null_dists, null_links = self._generate_null_dists(returns)
    k_range = list(range(1, k_max+1))

    for k in k_range:
      clusters = fcluster(Z, t=k, criterion="maxclust")

      W_k_real = self._compute_cl_dispersion(dist_mtx, clusters)
      log_disp_k = np.log(max(W_k_real, 1e-300))

      E_W_k, s_k = self._compute_ref_log_disp(null_dists, null_links, k)

      Gap_k.append(E_W_k - log_disp_k)
      s_k_list.append(s_k)

    gaps = np.array(Gap_k)
    s_k_list = np.array(s_k_list)

    optimal_k = k_range[np.argmax(gaps)]
    for idx in range(len(k_range)-1):
      if gaps[idx] >= gaps[idx+1] - s_k_list[idx+1]:
        optimal_k = k_range[idx]
        break

    optimal_k = max(optimal_k, 2) if dist_mtx.shape[0] > 1 else optimal_k
    return int(optimal_k)


  def get_clusters(self, dist_mtx:pd.DataFrame, returns:pd.DataFrame) -> pd.DataFrame:
    comp_disp = squareform(dist_mtx, checks=False)
    Z = linkage(comp_disp, method="single")

    optimal_k = self._get_k_clusters(dist_mtx, Z, returns)
    labels = fcluster(Z, t=optimal_k, criterion="maxclust")

    return pd.DataFrame({'Asset': returns.columns, 'Cluster': labels}), Z


  def _order_cluster_ids(self, Z, clusters_df):
    label_by_asset = clusters_df.set_index("Asset")["Cluster"]
    leaf_order = leaves_list(Z)

    asset_order = clusters_df["Asset"].values[leaf_order]
    ordered_labels = label_by_asset.loc[asset_order].values

    seen, ordered_ids = set(), []
    for lbl in ordered_labels:
      if lbl not in seen:
        seen.add(lbl)
        ordered_ids.append(lbl)

    return ordered_ids

In [34]:
class Optimizer(CVaREngine):
  def __init__(self, params, debug=False, **kwargs):
    super().__init__(params=params, debug=debug, **kwargs)
    self.p = params
    self.debug = debug
    self.lam = 1

  def _calculate_w(self, x):
    x_opt = np.asarray(x.value).ravel()
    w = x_opt/x_opt.sum()
    return w

  def _get_intra_cluster_w(self, cluster_R, cluster_idx, risk_budget=None):
    S, N = cluster_R.shape
    x = cp.Variable(N, pos=True)

    if risk_budget is None:
      b = np.ones(N)/N
    else:
      b = np.asarray(risk_budget, dtype=float)
      b = b/np.sum(b)

    zeta = cp.Variable()

    cluster_L = -(cluster_R@x)
    excess = cp.pos(cluster_L - zeta)

    cvar = zeta + cp.mean(excess)/(1-self.p.cvar_q)

    barrier = -self.lam*cp.sum(cp.multiply(b, cp.log(x)))

    obj = cp.Minimize(cvar + barrier)

    constraints = [
      x >= 0,
      x <= 1e+2
    ]

    prob = cp.Problem(obj, constraints)
    prob.solve(cp.CLARABEL)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise ValueError(f"Optimization failed: {prob.status}")

    w = self._calculate_w(x.value)
    _, cvar_p = self.compute_cvar(cluster_R, w)

    return w, cvar_p

